In [1]:
import pandas as pd
import requests
import numpy as np
from bs4 import BeautifulSoup
import json, os, time, pdb
import sys
import warnings, logging
import itertools
from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, WebDriverException
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import datetime

In [2]:
# Example RugbyPass Data URLs
# https://www.rugbypass.com/live/gwent-dragons-vs-munster/?g=947202
# https://www.rugbypass.com/live/gwent-dragons-vs-munster/stats/?g=947202
# https://www.rugbypass.com/live/gwent-dragons-vs-munster/head-to-head/?g=947202

# 

## RugbyPass Stats Data Sections 

### Match Summary 

In [3]:
test_url = "https://www.rugbypass.com/live/gwent-dragons-vs-munster/stats/?g=947202"

resp = requests.get(test_url).content
soup = BeautifulSoup(resp, 'html.parser')

In [4]:
stats_summ_table = soup.find(id='statsSummary_stats')
stats_summ_table.text

'   1 Penalty Goals 2     3 Tries 2     2 Conversions 2     0 Drop Goals 0     141 Carries 78     0 Line Breaks 2     11 Turnovers Lost 9     4 Turnovers Won 3   '

In [5]:
def format_row_data(div_content):
    row_lst = div_content.text.strip().split(' ')
    metric_dict = {
        'metric_name': [('').join(row_lst[1:-1])],
        'home_val': [row_lst[0]],
        'away_val': [row_lst[-1]]
    }
    if metric_dict['metric_name'] == ['']:
        return None
    else:
        return metric_dict


In [6]:
format_row_data(stats_summ_table.find_all('div')[6])

{'metric_name': ['Tries'], 'home_val': ['3'], 'away_val': ['2']}

In [7]:
[ i for i in list(map(format_row_data, stats_summ_table.find_all('div'))) if i is not None]

[{'metric_name': ['PenaltyGoals'], 'home_val': ['1'], 'away_val': ['2']},
 {'metric_name': ['Tries'], 'home_val': ['3'], 'away_val': ['2']},
 {'metric_name': ['Conversions'], 'home_val': ['2'], 'away_val': ['2']},
 {'metric_name': ['DropGoals'], 'home_val': ['0'], 'away_val': ['0']},
 {'metric_name': ['Carries'], 'home_val': ['141'], 'away_val': ['78']},
 {'metric_name': ['LineBreaks'], 'home_val': ['0'], 'away_val': ['2']},
 {'metric_name': ['TurnoversLost'], 'home_val': ['11'], 'away_val': ['9']},
 {'metric_name': ['TurnoversWon'], 'home_val': ['4'], 'away_val': ['3']}]

### Territory

In [8]:
# Total Field Territory 
# Percentage of match spent in each quadrant of the field Horizontally 
# 4 values for the mach on the whole

terr_base = soup.find(id='territory_stats').text

terr_lst = [i for i in terr_base.strip().split(' ') if i != '']
quad_stats = terr_lst[0].split('%')

home_total_terr = terr_lst[1]
away_total_terr = terr_lst[-1]
 


In [9]:
player_margin = soup.find(class_='player-stats-container tab-content')

In [10]:

# from playwright.sync_api import sync_playwright

# with sync_playwright() as p:
#     browser = p.chromium.launch(headless=False)  # Set headless=True for background
#     page = browser.new_page()
#     page.goto(test_url)
#     print(page.title())
#     browser.close()


import asyncio
from playwright.async_api import async_playwright

async def scrape():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(test_url)
        
        # wait for content to load, e.g.:
        print(page.title())
        # await page.wait_for_selector('.some-selector')
        content = await page.content()
        
        await browser.close()
        return content

content = await scrape()  

TimeoutError: Page.goto: Timeout 30000ms exceeded.
Call log:
  - navigating to "https://www.rugbypass.com/live/gwent-dragons-vs-munster/stats/?g=947202", waiting until "load"


In [22]:
from playwright.async_api import async_playwright
import time 
TARGET_URL = test_url # swap in your target URL
DIV_CLASS = "players"               # the class you want to click

async def test_click(url, div_class):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)  # headless=False so you can watch it
        page = await browser.new_page(viewport={"width": 1420, "height": 1500})
        await page.wait_for_timeout(3000)
        # Navigate and wait until network is idle (good for JS-heavy sites)
        await page.goto(url, wait_until="load", timeout=90000)
        # await page.wait_for_timeout(1000)

        # Wait for the target element to appear before clicking
        # target = await page.wait_for_selector(f"div.{div_class}", timeout=10000)

        target = await page.wait_for_selector("span.title:has-text('Carries')", timeout=10000)
        time.sleep(2)
# await target.click()
        await target.click()

        page.evalute("window.scrollBy(0, window.innerHeight);")
        await page.wait_for_timeout(1000)


        dd_target = await page.wait_for_selector("div.filter stats")
        await dd_target.click()
        # print(page.inner_html("div.scroll-box"))
        time.sleep(3)
        # Give the page a moment to react, then grab the HTML
        await page.wait_for_load_state("networkidle")
        content = await page.content()

        await browser.close()
        return content

content = await test_click(TARGET_URL, DIV_CLASS)
print(content[:2000])  # preview first 2000 chars

AttributeError: 'Page' object has no attribute 'evalute'

In [17]:
print(soup.find(class_='rs').prettify())

<div class="rs">
 <div class="tab-box">
  <div class="tabs">
   <div class="tab selected" data-id="teams" onclick="MatchCentreTeams.selectTab(this)">
    Teams
   </div>
   <div class="tab" data-id="players" onclick="MatchCentreTeams.selectTab(this)">
    Players
   </div>
  </div>
 </div>
 <div class="custom-banner live stats">
  <iframe src="https://rugbypass.space/banner/21?country=US&amp;tracking=dXRtX3NvdXJjZT1ycF9zaXRlJnV0bV9tZWRpdW09cnBfbmF0aXZlX2FkJnV0bV9jYW1wYWlnbj1XU0ZfU1VSVkVZJnV0bV9jb250ZW50PVJQLVMtVVNBLVRvcF9Vbml0&amp;height=120§ion=live&amp;path=YjRXdFh1bUVEamJxODVFU1Nwc0JmcVhOWWpTSGltcE1ibk9Za1RHcW5GdTVMUXRPNHFkMkR3c0V5bFlESUdKRjRCUDFNTFFFREF6R0ZsdS9pMS9aNmlQYXpIbS82cTFFY0JDNTZVL2IyWjY5QWw2MkpndURMQlVkeE9YcFhEaWRYMDFVSHZLVzlxTklZek85SGN2dGI4Q2ZoQUw4RHNBPQ==" title="RugbyPass and RugbyPass TV content">
  </iframe>
 </div>
 <div class="player-stats-container tab-content" data-id="players">
  <div class="player-stats-selector">
   <div class="selector-back">
    <img alt="B

In [10]:
soup.find(class_='players-stats components-box selector')

In [14]:
soup.find(class_='player-stats-container tab-content show-selector').text

AttributeError: 'NoneType' object has no attribute 'text'

In [11]:
print(len(soup.find_all(class_='player-stats-selector')))

1


In [12]:
print(player_margin.find(class_='players').prettify())

<div class="players no-pad" id="players-selector-result">
 <div class="player odd">
  <div class="num">
   1
  </div>
  <div class="logo">
   <img alt="Munster" class="lazy png" data-src="https://eu-cdn.rugbypass.com/webp-images/images/team-images/logos/png/310.png.webp?maxw=40&amp;v=1762927734" data-srcset="https://eu-cdn.rugbypass.com/webp-images/images/team-images/logos/png/310.png.webp?maxw=80&amp;v=1762927734 2x"/>
  </div>
  <div class="name">
   Brian Gleeson
  </div>
  <div class="total">
   19
  </div>
 </div>
 <div class="player even">
  <div class="num">
   2
  </div>
  <div class="logo">
   <img alt="Munster" class="lazy png" data-src="https://eu-cdn.rugbypass.com/webp-images/images/team-images/logos/png/310.png.webp?maxw=40&amp;v=1762927734" data-srcset="https://eu-cdn.rugbypass.com/webp-images/images/team-images/logos/png/310.png.webp?maxw=80&amp;v=1762927734 2x"/>
  </div>
  <div class="name">
   Diarmuid Barron
  </div>
  <div class="total">
   14
  </div>
 </div>
 <div

In [11]:
print(player_margin.prettify())

<div class="player-stats-container tab-content" data-id="players">
 <div class="player-stats-selector">
  <div class="selector-back">
   <img alt="Back" class="cta lazy" data-src="https://eu-cdn.rugbypass.com/images/match-centre/components/cta-arrow.png" data-srcset="https://eu-cdn.rugbypass.com/images/match-centre/components/cta-arrow.png 2x"/>
   Back
  </div>
  <section class="players-stats component-box selector">
   <div class="filters">
    <div class="filter stats">
     <span class="label">
      Carries
     </span>
     <img class="arrow lazy" data-src="https://eu-cdn.rugbypass.com/images/common/iconsMenuDdArrowDown-black.png"/>
     <div class="list">
      <div class="scroll-box">
       <div class="title">
        Top Stats:
       </div>
       <div class="list-item selected" data-id="carries" data-type="stat">
        Carries
       </div>
       <div class="list-item" data-id="clean_breaks" data-type="stat">
        Line Breaks
       </div>
       <div class="list-item

In [10]:
len(stats_summ_table.find_all('div'))

48

In [21]:
48/6

8.0

In [18]:
print(stats_summ_table.find_all('div')[6].prettify())

<div class="stat">
 <div class="line lh">
 </div>
 <div>
  3
 </div>
 <div class="mid smaller">
  Tries
 </div>
 <div>
  2
 </div>
 <div class="line">
 </div>
</div>



In [19]:
print(stats_summ_table.find_all('div')[12].prettify())

<div class="stat">
 <div class="line">
 </div>
 <div>
  2
 </div>
 <div class="mid smaller">
  Conversions
 </div>
 <div>
  2
 </div>
 <div class="line">
 </div>
</div>



In [7]:
print(stats_summ_table.prettify())

<div class="stats last-radius" id="statsSummary_stats">
 <div class="stat">
  <div class="line">
  </div>
  <div>
   1
  </div>
  <div class="mid smaller">
   Penalty Goals
  </div>
  <div>
   2
  </div>
  <div class="line rh">
  </div>
 </div>
 <div class="stat">
  <div class="line lh">
  </div>
  <div>
   3
  </div>
  <div class="mid smaller">
   Tries
  </div>
  <div>
   2
  </div>
  <div class="line">
  </div>
 </div>
 <div class="stat">
  <div class="line">
  </div>
  <div>
   2
  </div>
  <div class="mid smaller">
   Conversions
  </div>
  <div>
   2
  </div>
  <div class="line">
  </div>
 </div>
 <div class="stat">
  <div class="line">
  </div>
  <div>
   0
  </div>
  <div class="mid smaller">
   Drop Goals
  </div>
  <div>
   0
  </div>
  <div class="line">
  </div>
 </div>
 <div class="stat">
  <div class="line lh">
  </div>
  <div>
   141
  </div>
  <div class="mid smaller">
   Carries
  </div>
  <div>
   78
  </div>
  <div class="line">
  </div>
 </div>
 <div class="stat">
 

In [5]:
soup.find(id='territory_stats')

<div id="territory_stats"> <div class="field"> <img class="left-posts posts lazy" data-src="https://eu-cdn.rugbypass.com/images/match-centre/components/left-posts.png" data-srcset="https://eu-cdn.rugbypass.com/images/match-centre/components/left-posts@2x.png 2x"/> <img class="right-posts posts lazy" data-src="https://eu-cdn.rugbypass.com/images/match-centre/components/right-posts.png" data-srcset="https://eu-cdn.rugbypass.com/images/match-centre/components/right-posts@2x.png 2x"/> <div class="percents" id="territory_canvas_percents"> <div data-index="0">12%</div><div data-index="1">26%</div><div data-index="2">34%</div><div data-index="3">27%</div> </div> <canvas height="116" id="territory_canvas" width="352"></canvas> <script> window.scriptsToInit.push('GraphCanvas.set("territory_canvas",[12,26,34,27]);'); </script> </div> <div class="stats last-radius"> <div class="stat nb"> <div><img alt="Munster" class="lazy png" data-src="https://eu-cdn.rugbypass.com/webp-images/images/team-images

In [6]:
print(soup.prettify())

<!DOCTYPE html>
<html itemscope="" itemtype="http://schema.org/NewsMediaOrganization" lang="en">
 <head>
  <script>
   (function (w, d, s, l, i) { w[l] = w[l] || []; w[l].push({ 'gtm.start': new Date().getTime(), event: 'gtm.js' }); var f = d.getElementsByTagName(s)[0], j = d.createElement(s), dl = l != 'dataLayer' ? '&l=' + l : ''; j.async = true; j.src = 'https://www.googletagmanager.com/gtm.js?id=' + i + dl; f.parentNode.insertBefore(j, f); })(window, document, 'script', 'dataLayer', 'GTM-TBRKPLB'); window.dataLayer = window.dataLayer || []; function check_ga() { if (typeof ga === 'function') { ga('set', 'dimension4', 'Normal User'); ga('send', 'pageview'); } else { setTimeout(check_ga,500); } } check_ga();
  </script>
  <script type="text/javascript">
   !(function(o,n,t){t=o.createElement(n),o=o.getElementsByTagName(n)[0],t.async=1,t.src="https://annoyedairport.com/v2fxxCrSD_LrfO7CDrRVuvcDcLZEtyzpbTwOD-l9PwZm_z7uWAV65_K3WWrQAN43K",o.parentNode.insertBefore(t,o)})(document,"script"

### Territory 
### Possession 
### Set Plays 
### Attack 
### Turnovers 
### Penalties 
### Defence 
### Kicks 